# 15.4 数据污染检测 (Data Contamination Detection)

> 🕐 预估学习时间：30分钟

数据污染检测是确保LLM评估结果可信的关键步骤。当评估基准的数据出现在训练集中时，模型可能"记住"答案而非真正学会任务，导致评估分数虚高。

本节介绍数据污染的检测方法，包括N-gram重叠、Embedding相似度、成员推断等，并讨论污染缓解策略。

## 1. 数据污染概述

数据污染（Data Contamination）指评估基准中的样本出现在模型训练数据中，导致评估结果失真。

### 污染类型
| 类型 | 定义 | 影响 |
|------|------|------|
| 输入污染 | 基准的输入出现在训练集 | 轻微，模型可能熟悉输入模式 |
| 输出污染 | 基准的标签/答案出现在训练集 | 严重，模型直接记住答案 |
| 完整污染 | 输入+输出都出现在训练集 | 极严重，评估完全失效 |

### 污染来源
- **网络爬取**：Common Crawl等爬取的数据包含基准数据
- **开源数据集**：训练集与评估集来源重叠
- **人工标注**：标注者复制基准内容到训练数据
- **多轮训练**：上一轮评估数据泄漏到下一轮训练

### 对评估的影响
- 评估分数虚高，无法反映真实能力
- 模型间对比不公平
- 误导研究方向，过度优化污染基准

In [ ]:
import torch
import numpy as np
import re
from collections import Counter, defaultdict

torch.manual_seed(42)

class ContaminationDetector:
    """数据污染检测基类，提供数据模拟与通用工具。"""
    def __init__(self, seed=42):
        self.seed = seed
        torch.manual_seed(seed)
        np.random.seed(seed)

    def simulate_corpus(self, n_docs=200, vocab_size=500, avg_len=40):
        """模拟训练语料：每个文档是token id序列。"""
        docs = []
        for _ in range(n_docs):
            length = int(torch.randint(avg_len // 2, avg_len * 2, (1,)).item())
            doc = torch.randint(0, vocab_size, (length,)).tolist()
            docs.append(doc)
        return docs

    def simulate_benchmark(self, n_samples=30, vocab_size=500, avg_len=20):
        """模拟评估基准：每个样本是(input, output)对。"""
        samples = []
        for _ in range(n_samples):
            in_len = int(torch.randint(avg_len // 2, avg_len, (1,)).item())
            out_len = int(torch.randint(5, avg_len // 2, (1,)).item())
            inp = torch.randint(0, vocab_size, (in_len,)).tolist()
            out = torch.randint(0, vocab_size, (out_len,)).tolist()
            samples.append({'input': inp, 'output': out})
        return samples

    def inject_contamination(self, corpus, benchmark, level='light'):
        """将基准数据注入训练语料，模拟污染。"""
        n_inject = {'clean': 0, 'light': len(benchmark) // 3, 'heavy': len(benchmark)}[level]
        contaminated = list(corpus)
        injected_ids = list(range(n_inject))
        for i in injected_ids:
            sample = benchmark[i]
            contaminated.append(sample['input'] + sample['output'])
        return contaminated, injected_ids

detector = ContaminationDetector(seed=42)
benchmark = detector.simulate_benchmark(n_samples=30, vocab_size=500, avg_len=20)

scenarios = {}
for level in ['clean', 'light', 'heavy']:
    base_corpus = detector.simulate_corpus(n_docs=200, vocab_size=500, avg_len=40)
    corpus, injected = detector.inject_contamination(base_corpus, benchmark, level=level)
    scenarios[level] = {'corpus': corpus, 'injected': injected}

print('=== Data Contamination Simulation ===')
print(f'Benchmark samples: {len(benchmark)}')
for level, data in scenarios.items():
    n_inj = len(data['injected'])
    rate = n_inj / len(benchmark)
    n_corpus = len(data['corpus'])
    print(f'  {level:6s}: corpus={n_corpus} docs, injected={n_inj} ({rate:.0%})')

print(f'\nKey: Contamination level controls how many benchmark samples leak into training.')
print(f'Clean=0%, Light≈33%, Heavy=100% injection ratios simulate real-world scenarios.')

## 2. N-gram 重叠检测

N-gram重叠是最直接的污染检测方法：从基准数据提取n-gram，在训练语料中搜索匹配。

### 方法原理
1. 从基准样本提取n-gram（通常8-gram或13-gram）
2. 在训练语料中搜索这些n-gram
3. 计算重叠比例，超过阈值则判定为污染

### 参数选择
- **n=8**：检测短片段重叠，敏感但可能有误报
- **n=13**：检测长片段重叠，精确但可能漏报短污染
- **阈值**：通常10%以上重叠视为污染

In [ ]:
class NgramContaminationDetector(ContaminationDetector):
    """基于N-gram重叠的污染检测器。"""
    def __init__(self, n=8, seed=42):
        super().__init__(seed=seed)
        self.n = n

    def extract_ngrams(self, token_seq):
        """从token序列提取n-gram集合。"""
        if len(token_seq) < self.n:
            return set()
        return {tuple(token_seq[i:i+self.n]) for i in range(len(token_seq) - self.n + 1)}

    def build_corpus_ngram_index(self, corpus):
        """构建训练语料的n-gram索引。"""
        index = set()
        for doc in corpus:
            index |= self.extract_ngrams(doc)
        return index

    def detect(self, corpus, benchmark):
        """检测基准样本在训练语料中的n-gram重叠。"""
        corpus_index = self.build_corpus_ngram_index(corpus)
        results = []
        for i, sample in enumerate(benchmark):
            full_seq = sample['input'] + sample['output']
            sample_ngrams = self.extract_ngrams(full_seq)
            if not sample_ngrams:
                results.append({'idx': i, 'overlap': 0.0, 'matched': 0, 'total': 0})
                continue
            matched = len(sample_ngrams & corpus_index)
            overlap = matched / len(sample_ngrams)
            results.append({
                'idx': i, 'overlap': overlap,
                'matched': matched, 'total': len(sample_ngrams)
            })
        return results

    def summarize(self, results, threshold=0.1):
        """汇总检测结果。"""
        contaminated = [r for r in results if r['overlap'] > threshold]
        avg_overlap = sum(r['overlap'] for r in results) / len(results)
        return {
            'n_samples': len(results),
            'n_contaminated': len(contaminated),
            'contamination_rate': len(contaminated) / len(results),
            'avg_overlap': avg_overlap,
            'max_overlap': max(r['overlap'] for r in results),
        }

print('=== N-gram Overlap Detection ===')
for n in [8, 13]:
    print(f'\n--- {n}-gram detection ---')
    ng_detector = NgramContaminationDetector(n=n, seed=42)
    for level, data in scenarios.items():
        results = ng_detector.detect(data['corpus'], benchmark)
        summary = ng_detector.summarize(results)
        rate = summary['contamination_rate']
        avg = summary['avg_overlap']
        mx = summary['max_overlap']
        n_contam = summary['n_contaminated']
        n_total = summary['n_samples']
        print(f'  {level:6s}: contaminated={n_contam}/{n_total}, '
              f'rate={rate:.0%}, avg_overlap={avg:.3f}, max={mx:.3f}')

print(f'\nKey: N-gram overlap directly measures lexical duplication between benchmark and corpus.')
print(f'Longer n-grams (13) are more precise; shorter n-grams (8) catch partial contamination.')

## 3. Embedding 相似度检测

Embedding相似度检测通过语义向量比较基准与训练数据，能发现非字面的语义污染。

### 方法原理
1. 将基准样本和训练文档编码为向量
2. 计算基准与每个训练文档的余弦相似度
3. 取最大相似度或top-k平均作为污染分数

### 与N-gram对比
| 方面 | N-gram | Embedding |
|------|--------|-----------|
| 检测粒度 | 字面重叠 | 语义相似 |
| 抗改写 | 弱 | 强 |
| 计算成本 | 低 | 高 |
| 误报率 | 低 | 较高 |

In [ ]:
class EmbeddingContaminationDetector(ContaminationDetector):
    """基于Embedding相似度的污染检测器。"""
    def __init__(self, d=64, seed=42):
        super().__init__(seed=seed)
        self.d = d

    def encode(self, token_seq):
        """模拟将token序列编码为向量（实际用随机向量+池化）。"""
        if len(token_seq) == 0:
            return torch.zeros(self.d)
        tokens = torch.tensor(token_seq[-self.d:]) if len(token_seq) >= self.d else torch.tensor(token_seq)
        emb = torch.randn(len(tokens), self.d)
        return emb.mean(dim=0)

    def encode_corpus(self, corpus):
        """编码整个训练语料。"""
        return torch.stack([self.encode(doc) for doc in corpus])

    def detect(self, corpus, benchmark, top_k=3):
        """检测基准与训练语料的语义相似度。"""
        corpus_emb = self.encode_corpus(corpus)
        corpus_emb = torch.nn.functional.normalize(corpus_emb, dim=-1)
        results = []
        for i, sample in enumerate(benchmark):
            seq = sample['input'] + sample['output']
            emb = self.encode(seq)
            emb = torch.nn.functional.normalize(emb.unsqueeze(0), dim=-1)
            sims = (emb @ corpus_emb.T).squeeze(0)
            max_sim = sims.max().item()
            topk_avg = sims.topk(min(top_k, len(sims))).values.mean().item()
            results.append({
                'idx': i, 'max_sim': max_sim,
                'topk_avg': topk_avg, 'contaminated': max_sim > 0.7
            })
        return results

    def summarize(self, results, threshold=0.7):
        contaminated = [r for r in results if r['max_sim'] > threshold]
        avg_sim = sum(r['max_sim'] for r in results) / len(results)
        return {
            'n_samples': len(results),
            'n_contaminated': len(contaminated),
            'contamination_rate': len(contaminated) / len(results),
            'avg_max_sim': avg_sim,
        }

print('=== Embedding Similarity Detection ===')
emb_detector = EmbeddingContaminationDetector(d=64, seed=42)
for level, data in scenarios.items():
    results = emb_detector.detect(data['corpus'], benchmark)
    summary = emb_detector.summarize(results)
    rate = summary['contamination_rate']
    avg = summary['avg_max_sim']
    n_contam = summary['n_contaminated']
    n_total = summary['n_samples']
    print(f'  {level:6s}: contaminated={n_contam}/{n_total}, rate={rate:.0%}, avg_max_sim={avg:.3f}')

print(f'\nKey: Embedding similarity catches semantic contamination that N-gram misses.')
print(f'Higher threshold reduces false positives but may miss subtle contamination.')

## 4. 成员推断检测

成员推断（Membership Inference）通过模型对样本的置信度判断其是否在训练集中。

### 方法原理
1. 计算模型在基准样本上的loss/perplexity
2. 与参考集（未训练数据）的loss对比
3. 若基准loss显著低于参考集，可能存在污染

### 检测信号
- **Loss差异**：污染样本loss更低
- **置信度**：污染样本置信度更高
- **梯度范数**：污染样本梯度更小
- **输出熵**：污染样本熵更低

In [ ]:
class MembershipInferenceDetector(ContaminationDetector):
    """基于成员推断的污染检测器。"""
    def __init__(self, d=64, seed=42):
        super().__init__(seed=seed)
        self.d = d

    def simulate_model_loss(self, sample, seen=True):
        """模拟模型在样本上的loss（污染样本loss更低）。"""
        base = torch.randn(1).item()
        if seen:
            return max(0.05, abs(base) * 0.3 + 0.1)
        return abs(base) * 1.0 + 0.5

    def compute_losses(self, benchmark, reference_set, contaminated_idx):
        """计算基准与参考集的loss。"""
        results = []
        cont_set = set(contaminated_idx)
        for i, sample in enumerate(benchmark):
            seen = i in cont_set
            bench_loss = self.simulate_model_loss(sample, seen=seen)
            ref_loss = self.simulate_model_loss(sample, seen=False)
            results.append({
                'idx': i,
                'bench_loss': bench_loss,
                'ref_loss': ref_loss,
                'loss_ratio': bench_loss / ref_loss,
                'seen': seen,
            })
        return results

    def detect(self, results, threshold=0.6):
        """基于loss比率判定污染。"""
        flagged = [r for r in results if r['loss_ratio'] < threshold]
        return {
            'n_flagged': len(flagged),
            'flag_rate': len(flagged) / len(results),
            'avg_ratio': sum(r['loss_ratio'] for r in results) / len(results),
            'true_pos': sum(1 for r in flagged if r['seen']),
            'false_pos': sum(1 for r in flagged if not r['seen']),
        }

print('=== Membership Inference Detection ===')
mi_detector = MembershipInferenceDetector(d=64, seed=42)
reference_set = detector.simulate_benchmark(n_samples=30, vocab_size=500, avg_len=20)

for level, data in scenarios.items():
    results = mi_detector.compute_losses(benchmark, reference_set, data['injected'])
    summary = mi_detector.detect(results, threshold=0.6)
    rate = summary['flag_rate']
    avg = summary['avg_ratio']
    tp = summary['true_pos']
    fp = summary['false_pos']
    n_flagged = summary['n_flagged']
    n_inj = len(data['injected'])
    print(f'  {level:6s}: flagged={n_flagged}, rate={rate:.0%}, '
          f'avg_ratio={avg:.3f}, TP={tp}/{n_inj}, FP={fp}')

print(f'\nKey: Membership inference leverages model confidence as a contamination signal.')
print(f'Loss ratio < threshold indicates the model has likely seen the sample before.')

## 5. 污染缓解策略

检测到污染后，需要采取措施缓解其对评估的影响。

### 缓解方法
| 方法 | 描述 | 效果 |
|------|------|------|
| 去重 | 从训练集移除与基准重叠的样本 | 根治，但成本高 |
| Canary字符串 | 在基准中插入标记，检测泄漏 | 早期预警 |
| 基准隔离 | 训练时过滤基准来源域名 | 预防性 |
| 评估修正 | 对污染样本降权或排除 | 事后补救 |
| 动态基准 | 定期更新评估集 | 长期有效 |

In [ ]:
class ContaminationMitigator:
    """数据污染缓解器。"""
    def __init__(self, n=13, seed=42):
        self.n = n
        self.detector = NgramContaminationDetector(n=n, seed=seed)
        torch.manual_seed(seed)

    def deduplicate(self, corpus, benchmark, threshold=0.1):
        """从训练语料中移除与基准重叠的文档。"""
        bench_ngrams = set()
        for sample in benchmark:
            bench_ngrams |= self.detector.extract_ngrams(sample['input'] + sample['output'])
        kept = []
        removed = 0
        for doc in corpus:
            doc_ngrams = self.detector.extract_ngrams(doc)
            if not doc_ngrams:
                kept.append(doc)
                continue
            overlap = len(doc_ngrams & bench_ngrams) / len(doc_ngrams)
            if overlap > threshold:
                removed += 1
            else:
                kept.append(doc)
        return {'cleaned_corpus': kept, 'removed': removed, 'original_size': len(corpus)}

    def add_canary(self, benchmark, n_canaries=5, canary_len=16):
        """在基准样本中插入canary字符串用于泄漏检测。"""
        canarized = []
        canaries = []
        for i, sample in enumerate(benchmark):
            new_sample = {'input': list(sample['input']), 'output': list(sample['output'])}
            if i < n_canaries:
                canary = [1000 + i * 100 + j for j in range(canary_len)]
                new_sample['input'] = canary + new_sample['input']
                canaries.append({'idx': i, 'canary': canary})
            canarized.append(new_sample)
        return {'benchmark': canarized, 'canaries': canaries}

    def detect_canary_leak(self, corpus, canaries):
        """检测canary字符串是否泄漏到训练语料。"""
        corpus_text = ' '.join([' '.join(map(str, d)) for d in corpus])
        leaked = []
        for c in canaries:
            canary_text = ' '.join(map(str, c['canary']))
            if canary_text in corpus_text:
                leaked.append(c['idx'])
        return {'n_leaked': len(leaked), 'leaked_idx': leaked, 'n_total': len(canaries)}

    def evaluate_mitigation(self, original_corpus, cleaned_corpus, benchmark):
        """评估去重效果。"""
        before = self.detector.detect(original_corpus, benchmark)
        after = self.detector.detect(cleaned_corpus, benchmark)
        before_sum = self.detector.summarize(before)
        after_sum = self.detector.summarize(after)
        return {
            'before_rate': before_sum['contamination_rate'],
            'after_rate': after_sum['contamination_rate'],
            'reduction': before_sum['contamination_rate'] - after_sum['contamination_rate'],
        }

print('=== Contamination Mitigation ===')
mitigator = ContaminationMitigator(n=13, seed=42)

# 测试去重
heavy_corpus = scenarios['heavy']['corpus']
dedup_result = mitigator.deduplicate(heavy_corpus, benchmark, threshold=0.1)
orig_size = dedup_result['original_size']
n_removed = dedup_result['removed']
n_cleaned = len(dedup_result['cleaned_corpus'])
print(f'\n--- Deduplication ---')
print(f'  Original: {orig_size} docs')
print(f'  Removed: {n_removed} docs')
print(f'  Cleaned: {n_cleaned} docs')

# 测试canary
canary_result = mitigator.add_canary(benchmark, n_canaries=5, canary_len=16)
n_canaries = len(canary_result['canaries'])
print(f'\n--- Canary Strings ---')
print(f'  Added {n_canaries} canaries to benchmark')
leak_result = mitigator.detect_canary_leak(heavy_corpus, canary_result['canaries'])
n_leaked = leak_result['n_leaked']
n_total = leak_result['n_total']
print(f'  Leaked: {n_leaked}/{n_total}')

# 评估缓解效果
mit_eval = mitigator.evaluate_mitigation(heavy_corpus, dedup_result['cleaned_corpus'], benchmark)
before_rate = mit_eval['before_rate']
after_rate = mit_eval['after_rate']
reduction = mit_eval['reduction']
print(f'\n--- Mitigation Effect ---')
print(f'  Before: {before_rate:.0%} contaminated')
print(f'  After: {after_rate:.0%} contaminated')
print(f'  Reduction: {reduction:.0%}')

print(f'\nKey: Deduplication removes leaked samples; canaries provide early leak warning.')
print(f'Combining multiple mitigation strategies gives the best protection.')

## 📝 课后思考题

1. 数据污染检测的核心原理是什么？请用自己的话解释其工作过程。
2. N-gram重叠检测与Embedding相似度检测相比有哪些优势？存在什么局限性？
3. 在实际项目中，如何选择和配置成员推断检测相关的参数？需要考虑哪些因素？
4. 15.4 数据污染检测领域还有哪些前沿发展方向？结合本notebook的内容谈谈你的看法。

---
> 本节涵盖了15.4 数据污染检测的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。